##Environment Setup
This cell sets up the environment by mounting the Google Drive by defining the paths to the UCSD Anomlay Dataset zip folder. Then extracting the dataset if not already present. Printing path of Peds reveal both are accessible.

In [6]:
from google.colab import drive
from pathlib import Path
from zipfile import ZipFile

drive.mount('/content/drive')

DATA_ROOT = Path("/content/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset")
PED1_PATH = DATA_ROOT / "UCSDped1"
PED2_PATH = DATA_ROOT / "UCSDped2"

ZIP_PATH = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/UCSD_Anomaly_Dataset.zip")
EXTRACT_PATH = Path("/content/UCSD_Anomaly_Dataset")

if not (PED1_PATH.exists() and PED2_PATH.exists()):
  with ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_PATH)

print("Ped1:", PED1_PATH.exists(), "| Ped2:", PED2_PATH.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ped1: True | Ped2: True


In [7]:
from google.colab import userdata
from pathlib import Path

GH_TOKEN = userdata.get("GH_PAT")

REPO_DIR = Path(
    "/content/drive/MyDrive/SurveillanceAnomalyDetection/repo"
)

REPO_URL = (
    f"https://{GH_TOKEN}@github.com/"
    "Rishabh-G-Shetye/SurveillanceAnomalyDetection.git"
)

if not REPO_DIR.exists():
    !git clone {REPO_URL} "{REPO_DIR}"
    print("Repository cloned.")
else:
    print("Repository already exists — skipping clone.")

%cd "{REPO_DIR}"

Repository already exists — skipping clone.
/content/drive/MyDrive/SurveillanceAnomalyDetection/repo


##Sequence Inventory
Apart from importing the necessary libraries, it defines the helper functions to count the image frames and build an inventory of the dataset. It is made to iterate through the Train and Test splits of both the Ped1 and PEd2 datasets. It then records the information like the sequence name, number of frames and if ground truth masks exist.

In [8]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

IGNORE_NAMES = {".DS_Store", "._.DS_Store"}
FRAME_SUFFIXES = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}

def count_frames(seq_dir: Path) -> int:
  # Counting image frames in sequence folder
  return sum(
      1 for p in seq_dir.iterdir()
      if p.is_file() and p.suffix.lower() in FRAME_SUFFIXES
  )
def build_inventory(ped_path: Path, ped_name: str) -> list[dict]:
  #Walk Train/Test  fodler of on PEd dataset
  rows = []
  for split in ("Train", "Test"):
    split_path = ped_path / split
    for seq_dir in sorted(split_path.iterdir()):
      if not seq_dir.is_dir() or seq_dir.name in IGNORE_NAMES:
        continue
      if seq_dir.name.endswith("_gt"):
        continue
      gt_dir = split_path / f"{seq_dir.name}_gt"
      rows.append({
          "dataset": ped_name,
          "split": split,
          "sequence": seq_dir.name,
          "num_frames": count_frames(seq_dir),
          "has_gt": gt_dir.exists(),
          "num_gt_frames": count_frames(gt_dir) if gt_dir.exists() else 0,
      })
  return rows

inventory = build_inventory(PED1_PATH, "Ped1") + build_inventory(PED2_PATH, "Ped2")
df = pd.DataFrame(inventory)
df.head(10)

,dataset,split,sequence,num_frames,has_gt,num_gt_frames
0,Ped1,Train,Train001,200,False,0
1,Ped1,Train,Train002,200,False,0
2,Ped1,Train,Train003,200,False,0
3,Ped1,Train,Train004,200,False,0
4,Ped1,Train,Train005,200,False,0
5,Ped1,Train,Train006,200,False,0
6,Ped1,Train,Train007,200,False,0
7,Ped1,Train,Train008,200,False,0
8,Ped1,Train,Train009,200,False,0
9,Ped1,Train,Train010,200,False,0


##Sumary Statistics
This groups the inventory DataFrme by dataset and split to calculate the sumamry statistics such as number of sequences, total frames, average frames per sequence and the number of of sequences with ground truth masks.
The output provides the concise overview of the dataset's composition of both Ped1 and Ped2 distinguishing between the training and testing.

In [9]:
summary = df.groupby(["dataset", "split"]).agg(
    num_sequence=("sequence", "count"),
    total_frames=("num_frames", "sum"),
    avg_frames_per_seq=("num_frames", "mean"),
    num_with_gt=("has_gt", "sum")
).round(1)
summary

num_sequence  total_frames  avg_frames_per_seq  num_with_gt
dataset split                                                             
Ped1    Test             36          7200               200.0           10
        Train            34          6800               200.0            0
Ped2    Test             12          2010               167.5           12
        Train            16          2550               159.4            0

In [ ]:
%cd "{REPO_DIR}"
!git add notebooks/02_ucsd_dataset_exploration.ipynb
!git commit -m "Add dataset exploration: sequence inventory + summary stats"
